In [21]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')
print(os.getcwd()) 

C:\Users\Klara\retail-intelligence


In [22]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

folds = pd.read_csv('data/processed/walkforward_folds.csv')

print(f"Total folds (all products): {len(folds)}")
print(f"Products: {folds['StockCode'].nunique()}")
print(folds['DataQualityFlag'].value_counts())

folds_ok = folds[folds['DataQualityFlag'] == 'OK'].copy()
print(f"\nFolds after excluding flagged products: {len(folds_ok)}")
print(f"Products after exclusion: {folds_ok['StockCode'].nunique()}")

Total folds (all products): 100
Products: 20
DataQualityFlag
OK                                       90
Low reliability - sparse/spike-driven    10
Name: count, dtype: int64

Folds after excluding flagged products: 90
Products after exclusion: 18


In [23]:
import pandas as pd
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

from src.module2_demand.prophet_model import prepare_prophet_data

# Reload the raw weekly demand data, same as walk_forward_validation.py does
demand = pd.read_csv('data/processed/weekly_demand.csv')
demand['Week'] = pd.to_datetime(demand['Week'])

# Rebuild the prophet_df for 23843 exactly like the pipeline does
prophet_df = prepare_prophet_data(demand, '23843')

# Fold 5's test window was 2011-11-14 to 2011-12-05 — the last 4 weeks
test_fold5 = prophet_df.iloc[-4:]
print(test_fold5[['ds', 'y']])

Product 23843:
  Weeks of data: 54
  Date range: 2010-11-29 to 2011-12-05
  Mean weekly demand: 1499.9 units
  Max weekly demand: 80995 units
  Zero-demand weeks: 53
           ds      y
50 2011-11-14      0
51 2011-11-21      0
52 2011-11-28      0
53 2011-12-05  80995


In [24]:
watch_products = folds[folds['StockCode'].isin(['21977', '15036'])]
print(watch_products[['StockCode', 'Fold', 'Prophet_MAE', 'Naive_MAE', 
                        'Seasonal_MAE', 'BeatNaive', 'BeatSeasonal']].to_string(index=False))

print("\n21977 win rate vs naive:", 
      watch_products[watch_products['StockCode']=='21977']['BeatNaive'].mean())
print("15036 win rate vs naive:", 
      watch_products[watch_products['StockCode']=='15036']['BeatNaive'].mean())

StockCode  Fold  Prophet_MAE  Naive_MAE  Seasonal_MAE  BeatNaive  BeatSeasonal
    21977     1   306.498920     914.00        206.00       True         False
    21977     2   207.000000     253.00        362.00       True          True
    21977     3   355.737666     398.50        205.50       True         False
    21977     4   480.022832      43.75        104.75      False         False
    21977     5   237.550099      86.25         68.00      False         False
    15036     1   596.986919     288.00        198.00      False         False
    15036     2   616.055398      27.00         33.00      False         False
    15036     3   350.024397     225.00        213.00      False         False
    15036     4   347.965284     126.00         24.00      False         False
    15036     5   221.389880      66.00         63.00      False         False

21977 win rate vs naive: 0.6
15036 win rate vs naive: 0.0


In [25]:
fig = px.box(
    folds_ok,
    x='StockCode',
    y='Prophet_MAE',
    title='Prophet MAE Distribution Across Folds per Product (OK products only)',
    labels={'Prophet_MAE': 'MAE (units)', 'StockCode': 'Product'}
)
fig.show()

In [26]:
win_rate_summary = folds_ok.groupby('StockCode')['BeatNaive'].mean().reset_index()
win_rate_summary.columns = ['StockCode', 'WinRateVsNaive']

fig2 = px.bar(
    win_rate_summary.sort_values('WinRateVsNaive'),
    x='StockCode',
    y='WinRateVsNaive',
    title='Prophet Win Rate vs Naive Baseline (per product, OK only)',
    labels={'WinRateVsNaive': 'Win Rate (% of folds)'},
    color='WinRateVsNaive',
    color_continuous_scale='RdYlGn'
)
fig2.update_layout(yaxis_tickformat='.0%')
fig2.show()

In [27]:
fig3 = px.line(
    folds_ok.groupby('Fold')[['Prophet_MAE', 'Naive_MAE', 'Seasonal_MAE']].mean().reset_index(),
    x='Fold',
    y=['Prophet_MAE', 'Naive_MAE', 'Seasonal_MAE'],
    title='Average MAE by Fold — OK products only (does performance degrade over time?)',
    labels={'value': 'MAE', 'variable': 'Model'}
)
fig3.show()

# Week 5 — Day 1 Observations (Walk-Forward Validation)

## Data Quality Validation

**DataQualityFlag distribution:**

Out of 100 total folds (20 products × 5 folds), 90 folds were flagged as **OK** and 10 folds were flagged as **Low reliability**. The low-reliability folds belong exclusively to products **23843** and **23166** (5 folds each), confirming that the quality flags from Week 4 carried over correctly into the walk-forward framework.

All aggregate statistics, visualizations, and conclusions below use only products flagged as **OK** in order to avoid distortion from sparse and spike-driven demand patterns.

---

## Sparse Product Investigation

### Product 23843

Product 23843 remains the clearest example of why sparse products are problematic for standard forecasting models.

**Demand characteristics:**

* Mean weekly demand: 1,499.9 units
* Maximum weekly demand: 80,995 units
* Zero-demand weeks: 53 out of 54

The fifth test window contained three weeks with zero demand followed by a single 80,995-unit purchase. Both Prophet and the naive baseline predicted values close to zero and produced the exact same error (MAE = 20,248.8).

This confirms that the tie is not caused by a bug in the implementation but by the underlying demand pattern itself. Products dominated by rare bulk purchases violate the assumptions of traditional time-series models and should be treated separately from normal retail products.

---

## Product-Level Performance Analysis

### Product 21977

Week 4 initially suggested that product 21977 might require tuning due to poor performance in a single test window. Walk-forward validation tells a different story.

Results across five folds show that Prophet beats the naive baseline in **3 out of 5 folds (60%)**, indicating that the original concern was caused by a temporary test-window effect rather than a persistent forecasting problem.

No immediate tuning appears necessary.

### Product 15036

Product 15036 loses against the naive baseline in **all five folds (0% win rate)**.

Unlike 21977, this pattern is stable across multiple time windows and therefore represents genuine model underperformance rather than random variation.

Potential future improvements include:

* tuning `changepoint_prior_scale`;
* testing logistic growth with a capacity limit;
* introducing product-specific model configurations.

### Product 22086

Product 22086 shows the same behavior as 15036, losing against the naive baseline in all five folds.

This issue was not detected during Week 4's single-window evaluation and only became visible after applying walk-forward validation.

Product 22086 should be added to the watch list for future model tuning experiments.

---

## Fold-Level Performance

### Is Prophet consistently better than the baselines?

Performance varies significantly across both products and folds.

Products such as **85123A**, **21212**, **22386**, **84946**, and **85099F** show relatively stable forecasting errors across all five windows, while product **22197** exhibits much larger variation, with MAE values ranging from approximately 400 to 950 units.

At the aggregate level, Prophet alternates between outperforming and underperforming the naive baseline:

* Prophet wins in folds 1, 2, and 5;
* Prophet loses in folds 3 and 4.

This suggests that forecasting performance depends more on the demand characteristics of individual products than on any particular time period.

---

### Does performance degrade over time?

No consistent degradation pattern is visible.

Forecast errors do not increase steadily across later folds, and Prophet remains competitive in the final validation window.

The losses observed in folds 3 and 4 appear to be specific to the demand patterns present in those windows rather than evidence that the model deteriorates over time.

---

### Does Prophet consistently beat seasonal naive?

Yes.

Unlike the comparison against the simple naive baseline, Prophet outperforms the seasonal naive baseline in every fold.

This is the most stable result in the entire backtest and suggests that Prophet captures long-term demand trends more effectively than seasonal repetition alone.

---

## Key Conclusions

1. Walk-forward validation confirmed that products 23843 and 23166 remain genuinely low-reliability products.

2. Sparse, spike-driven demand continues to be the largest challenge for standard forecasting models.

3. Product-specific behavior has a much greater impact on forecasting accuracy than the choice of validation window.

4. Product 21977 was incorrectly flagged as problematic in Week 4 and does not require immediate tuning.

5. Products 15036 and 22086 consistently underperform and should be prioritized for future model improvements.

6. Prophet does not consistently outperform the naive baseline, but it remains reliably stronger than the seasonal naive benchmark.
